<a href="https://colab.research.google.com/github/jimhopgtu/google-ai-agents-daily-content-tool/blob/main/Daily_Relevant_Content.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [29]:
from google.adk.agents import Agent, SequentialAgent, ParallelAgent, LoopAgent
from google.adk.models.google_llm import Gemini
from google.adk.runners import InMemoryRunner
from google.adk.tools.google_search_tool import GoogleSearchTool
from google.adk.tools import FunctionTool
from google.genai import types
import json
from pydantic import BaseModel, Field, AliasChoices
from typing import List, Optional
import os
from google.colab import userdata, drive
from datetime import datetime
import pytz
eastern_tz = pytz.timezone('America/New_York')


print("✅ ADK components imported successfully.")

✅ ADK components imported successfully.


In [30]:

# This pulls the secret you just created and sets it as an environment variable
# Most Google SDKs (including ADK) look for "GOOGLE_API_KEY" automatically.
os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')

print("✅ API Key successfully loaded into environment!")

# from IPython.core.display import display, HTML
retry_config=types.HttpRetryOptions(
    attempts=5,  # Maximum retry attempts
    exp_base=7,  # Delay multiplier
    initial_delay=1, # Initial delay before first retry (in seconds)
    http_status_codes=[429, 500, 503, 504] # Retry on these HTTP errors
)


# Ensure Drive is mounted correctly
drive.mount('/content/drive', force_remount=True)


HISTORY_FILE = "/content/drive/MyDrive/AI/Google AI Agents/evaluated_articles.json"

def get_historical_urls():
    """Loads the list of URLs already processed in previous runs."""
    if os.path.exists(HISTORY_FILE):
        try:
            with open(HISTORY_FILE, 'r') as f:
                return set(json.load(f))
        except Exception as e:
            print(f"⚠️ Could not read history file, starting fresh: {e}")
            return set()
    return set()

def update_history_file(new_urls):
    """Appends new URLs to the persistent history file."""
    existing = get_historical_urls()
    existing.update(new_urls)
    with open(HISTORY_FILE, 'w') as f:
        json.dump(list(existing), f)
    print(f"💾 History updated: {len(existing)} total articles now tracked.")

# Load history at the start of the session
historical_seen_urls = get_historical_urls()
print(f"✅ Loaded {len(historical_seen_urls)} historical articles from Drive.")

print("Executed at:", datetime.now(eastern_tz))

✅ API Key successfully loaded into environment!
Mounted at /content/drive
✅ Loaded 0 historical articles from Drive.
Executed at: 2025-12-23 10:57:04.220380-05:00


In [38]:
search_instruction = """
You are a Research Scout for an Analytics Leader.
Your goal is to provide a balanced mix of content . For every run, you MUST use the search tool to find:

1. THE LATEST (24h): Top 3 industry-shifting news (e.g., Anthropic, OpenAI, Google).
2. THE ARCHITECTURE: Top 1 technical blog posts from engineering-heavy companies
   (e.g., MongoDB, Pinecone, Meta Engineering) that discuss 'why' or 'how'—not just 'what'.
3. THE STACK: Top 2 recent updates from the modern BI stack (e.g., dbt, Snowflake, Databricks, BigQuery, Looker, Atlan, PowerBI, Tableau)
4. THE LOCAL: 1 AI event in the NYC area or a major remote global summit.
5. Top 1 Obsidian plug in or use case that are new and could be useful.

## CRITICAL CONSTRAINTS
<Constraints>
- ANTI-FLUFF: If results are 'marketing fluff', refine keywords to include 'technical deep dive'.
- OUTPUT FORMAT: Return ONLY the URL and a 1-2 sentence summary. No full articles.
- MULTIMEDIA: YouTube videos are acceptable.
</Constraints>

## OUTPUT FORMAT (MANDATORY)
For every single item you find, you MUST follow this exact format:
- **Title**: [Name of the article/video]
- **Source URL**: [Insert the full direct link here]
- **Summary**: [1-2 sentences of why this matters for a data leader]
- **Date Published**:  [Insert the date here]

## CRITICAL RULES
- NEVER provide a news item without a corresponding URL.
- If you find a great story but the URL is missing from your tool output, do not include the story.
"""

# Define the data structure as a Python list/dictionary
LEADER_CONTEXT_DATA = {
    "ANALYTICS_LEADER_CONTEXT": [
        {
            "Category": "Modeling",
            "Shift": "Causal Inference (MMM/MTA), Advanced Models",
            "Action": "Prioritize Experimentation and Causal Strategy (A/B testing, incrementality)"
        },
        {
            "Category": "Architecture",
            "Shift": "Semantic Layer is SOT, often led by **Knowledge Engineer**",
            "Action": "Architect **AI Trust** and robust **Data Governance**"
        },
        {
            "Category": "Role & Skills",
            "Shift": "Analyst as Prompt Engineer/Consultant",
            "Action": "Coach for **Business Acumen** (the 'Why') and focus on recommendations"
        },
        {
            "Category": "Specialization",
            "Shift": "Deep expertise in one major stack (e.g., GCP, Azure)",
            "Action": "Standardize and Optimize the chosen stack to maximize value"
        },
        {
            "Category": "Analyst Profile",
            "Shift": "**Hybrid Role** (Business SME + Data Engineering + **Knowledge Engineering**)",
            "Action": "Redefine career path, mandate **DE fundamentals** and context structuring"
        },
        {
            "Category": "Governance",
            "Shift": "Mandatory focus on **Data Governance** and **AI TRiSM**",
            "Action": "Audit AI outputs, enforce lineage, and ensure ethical compliance"
        },
        {
            "Category": "Speed",
            "Shift": "Shift to **Real-Time** and **Edge Analytics**",
            "Action": "Invest in Modern Architecture (e.g., Data Mesh) for streaming data processing"
        },
        {
            "Category": "Leadership",
            "Shift": "Highest value in Human-Centric/Soft Skills and organizational influence",
            "Action": "Cultivate critical thinking, emotional intelligence, and **cross-department bridge-building** to remove data silos"
        }
    ]
}

# Convert it to a pretty-printed string ONLY when you need to feed it to the Agent
context_string = json.dumps(LEADER_CONTEXT_DATA, indent=2)
# Pass the current date into the instruction so the AI can calculate "recency"
current_date_str = datetime.now(eastern_tz).strftime("%B %d, %Y")


relevance_instruction = f"""
## ROLE
You are a Quality Controller for an Analytics Leader.
**TODAY'S DATE**: {current_date_str}

## YOUR TASK
1. **Analyze Content**: Review articles in 'raw_news_data' for technical depth and strategic relevance.
2. **Base Score (1-5)**: Assign an initial score based strictly on content quality.
3. **Recency Deduction (CRITICAL)**: Compare the article's publish date to TODAY'S DATE ({current_date_str}) and subtract points:
   - **Published > 7 days ago**: Subtract 1 point from Base Score.
   - **Published > 28 days ago**: Subtract 2 points from Base Score.
   - **Publish Date Unknown**: Subtract 1 point from Base Score.
4. **CRITICAL GATEKEEPER LOGIC**:
   - If an article scores **less than 3**, you MUST mark its status as "REJECTED".
   - Only articles with a score of 3, 4, or 5 should be marked as "APPROVED".
   - Discard "marketing fluff", generic AI hype, or basic tutorials.
5. **GOAL**: We need at least 6 high-quality (Score 3+) articles in total.

## OUTPUT FORMAT
Return a JSON list of objects with these keys: title, url, summary, score, category, date_published, and status.
"""

# This replaces 'ScoredArticle' to match the system expectations
class EvaluatedArticle(BaseModel):
    title: str
    # 'validation_alias' allows the AI to say 'url' OR 'source_url' without crashing
    url: str = Field(validation_alias=AliasChoices('url', 'source_url'))
    summary: str
    score: int
    # Making these 'Optional' with default values prevents "Missing Field" crashes
    category: Optional[str] = "General"
    date_published: Optional[str] = "N/A"

class RelevanceResponse(BaseModel):
    evaluated_articles: List[EvaluatedArticle]


print("Executed at:", datetime.now(eastern_tz))

Executed at: 2025-12-23 11:05:22.622581-05:00


In [39]:
# This is the function that the RefinerAgent will call to exit the loop.
def exit_loop():
    """Call this function ONLY when the critique is 'APPROVED', indicating the story is finished and no more changes are needed."""
    return {"status": "approved", "message": "Story approved. Exiting refinement loop."}


print("✅ exit_loop function created.")
print("Executed at:", datetime.now(eastern_tz))

✅ exit_loop function created.
Executed at: 2025-12-23 11:05:25.221099-05:00


In [43]:

# 1. Search Agent
search_agent = Agent(
    name="news_finder",
    model="gemini-2.0-flash", # Use 2.0 for speed/tool use gemini-2.0-flash gemini-1.5-flash or gemini-1.5-flash-8b
    tools=[GoogleSearchTool()],
    instruction=search_instruction,
    output_key="raw_news_data"
)

# 2. Relevance Agent
relevance_agent = Agent(
    name="relevance_evaluator",                  # 1. Added required 'name'
    model=Gemini(model_id="gemini-2.0-flash-exp"),
    instruction=relevance_instruction            # 2. Changed 'instructions' to 'instruction'
)

# 3. Loop Agent
story_refinement_loop = LoopAgent(
    name="StoryRefinementLoop",
    sub_agents=[search_agent, relevance_agent],
    # The SDK usually looks for a single condition string referencing the state
    max_iterations=3
)

# 4. Reporting Agent
reporting_agent = Agent(
    name="reporting_agent",
    model="gemini-2.0-flash",
    instruction="""
    ## TASK
    1. Read the articles stored in {{approved_stories?}}.
    2. If articles exist, format them into a professional Markdown report.

    ## FORMAT REQUIREMENTS
    - Use ## Headers for different categories.
    - For each article, use the following structure:
      ### [Title](URL)
      - **Score**: [Insert Score]/5
      - **Date Published**: [Insert Date]
      - **Key Takeaway**: [Insert Summary]
      - **Why this matters**: [Briefly explain strategic impact]
    - Ensure every Title is a clickable [Markdown Link](URL).
    """
)

print("Executed at:", datetime.now(eastern_tz))

Executed at: 2025-12-23 11:09:44.028679-05:00


In [44]:
APPROVED_STORIES_STORAGE = {"approved_stories": {"evaluated_articles": []}}
target_count = 5
iteration = 0
max_iterations = 3
session_seen_urls = set() # Tracks duplicates within THIS specific run

print(f"🚀 Starting Research Loop. Goal: {target_count} high-quality articles.")

while len(APPROVED_STORIES_STORAGE["approved_stories"]["evaluated_articles"]) < target_count and iteration < max_iterations:
    iteration += 1
    current_count = len(APPROVED_STORIES_STORAGE["approved_stories"]["evaluated_articles"])
    loop_session = f"daily_report_{pd.Timestamp.now().strftime('%H%M%S')}_{iteration}"

    print(f"\n--- 🔄 ITERATION {iteration} (Current Approved: {current_count}/{target_count}) ---")

    # STEP A: SEARCH - Pass BOTH historical and session URLs to the AI to avoid repeats
    all_known_urls = list(historical_seen_urls | session_seen_urls)

    runner = InMemoryRunner(agent=search_agent)
    print("🔍 Search Agent: Finding new articles...")
    search_events = await runner.run_debug(
        f"Find 6-10 unique news articles for an Analytics Leader. IGNORE these URLs as they were already processed: {all_known_urls[:10]}",
        session_id=loop_session
    )

    # STEP B: RELEVANCE & SCORING
    runner.agent = relevance_agent
    print("⚖️ Relevance Agent: Scoring content and filtering 'fluff'...")
    relevance_events = await runner.run_debug(
        "Analyze the new 'raw_news_data'. Only approve items with score >= 3.",
        session_id=loop_session
    )

    # STEP C: PYTHON-LEVEL FILTERING & HISTORY CHECK
    raw_ai_text = relevance_events[-1].content.parts[0].text
    clean_json = re.sub(r'^```json\s*|```$', '', raw_ai_text, flags=re.MULTILINE | re.DOTALL).strip()

    try:
        parsed_json = json.loads(clean_json)
        articles = parsed_json if isinstance(parsed_json, list) else parsed_json.get("evaluated_articles", [])

        new_approvals = 0
        for item in articles:
            url = item.get("url")
            score = item.get("score", 0)

            # CRITICAL CHECK: Ensure URL is not in history AND not already seen this session
            if score >= 3 and url not in historical_seen_urls and url not in session_seen_urls:
                APPROVED_STORIES_STORAGE["approved_stories"]["evaluated_articles"].append(item)
                session_seen_urls.add(url)
                new_approvals += 1
            elif url in historical_seen_urls:
                print(f"⏭️ Skipping (Already in History): {item.get('title')[:40]}...")

        print(f"✅ Added {new_approvals} qualified articles in this iteration.")

    except Exception as e:
        print(f"❌ Error parsing Relevance JSON: {e}")

# --- NEW: PERSIST THE NEWLY FOUND ARTICLES ---
if session_seen_urls:
    update_history_file(session_seen_urls)

# 3. FINAL REPORTING
final_vault = APPROVED_STORIES_STORAGE["approved_stories"]["evaluated_articles"]

if len(final_vault) >= 1:
    print(f"\n📊 Final Vault size: {len(final_vault)}. Generating Professional Report...")
    runner.agent = reporting_agent

    # We pass the EXACT filtered JSON to the reporter
    final_report_events = await runner.run_debug(
        f"Generate the markdown report using ONLY this data: {json.dumps({'evaluated_articles': final_vault})}",
        session_id="final_reporting_session"
    )
    print("\n✅ Report Generated Successfully.")
else:
    print("⚠️ Loop ended: Could not find enough high-quality content.")

# --- COST & TOKEN TRACKING ---
total_tokens = 0
for event in (search_events + relevance_events + (final_report_events if 'final_report_events' in locals() else [])):
    if hasattr(event, 'usage_metadata') and event.usage_metadata:
        total_tokens += event.usage_metadata.prompt_token_count
print(f"\n💰 Estimated tokens used: {total_tokens:,}")

🚀 Starting Research Loop. Goal: 5 high-quality articles.

--- 🔄 ITERATION 1 (Current Approved: 0/5) ---
🔍 Search Agent: Finding new articles...

 ### Created new session: daily_report_161003_1

User > Find 6-10 unique news articles for an Analytics Leader. IGNORE these URLs as they were already processed: []


news_finder > Here are some news articles for an Analytics Leader:

- **Title**: Anthropic, Google Gain Market Share At OpenAI's Expense In Enterprise API LLM Usage: Menlo Ventures Report
- **Source URL**: https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHOLN0hLXuVTjQEaE329w0eweavY683jXxt2HSRIeI6LW7K9puCqBpChTIBwc6nQO0jTvrmwCGqV7V1VL2xZalSVnJ1mpGyMJwBhn3_WUYztgB752SRc7HXlE3VZtqRK7YevZnjllds1DWjw2E5De7JdovFFFl4jf9HoPUkGlAzicr4-KqfzoNUzvAvaAyx-2n-bD9tNaAtR1Sqg1sqdAedYFhg0bYVRUW4z5CLfEU8x6UYMid3EPlLU73EWB8R
- **Summary**: A Menlo Ventures report indicates that OpenAI's market share in the enterprise LLM API space has declined significantly from 50% in 2023 to 27% in 2025, with Anthropic and Google gaining traction. Anthropic's market share surged to 40%, making it the leader in enterprise LLM API usage, while Google's share tripled to 21%.
- **Date Published**: 2025-12-23

- **Title**: Google, xAI, OpenAI sued over chatbot training
- **Source URL**: https://vertexaisea

In [45]:
# =========================================================
# UPDATED FINAL CELL: EXPORT TO DRIVE & UPDATE HISTORY
# =========================================================

# 1. Setup the Path
today_date = datetime.now(eastern_tz).strftime("%Y-%m-%d_%H%M")
filename = f"{today_date}_Analytics_Report.md"
save_path = "/content/drive/MyDrive/AI/Obsidian Vault/daily news/"

# Create folder if it's missing
if not os.path.exists(save_path):
    print(f"Creating missing directory: {save_path}")
    os.makedirs(save_path, exist_ok=True)

full_path = os.path.join(save_path, filename)

try:
    # FIX: Use 'final_report_events' (from Cell 8) instead of 'final_report'
    final_report_text = ""
    for event in reversed(final_report_events):
        if event.content and event.content.parts:
            text_parts = [p.text for p in event.content.parts if p.text]
            if text_parts:
                final_report_text = "\n".join(text_parts)
                break

    if not final_report_text:
        raise ValueError("No report content found in final_report_events")

    # Clean the markdown formatting
    clean_report = re.sub(r'^```(?:markdown)?\n?|```$', '', final_report_text.strip(), flags=re.MULTILINE)

    # Add metadata header
    header = f"""---
title: Analytics Leader Daily Report
date: {datetime.now(eastern_tz).strftime("%Y-%m-%d")}
generated: {datetime.now(eastern_tz).strftime("%Y-%m-%d %H:%M %Z")}
---

"""
    final_content = header + clean_report

    # Write the file and force a sync to Disk
    with open(full_path, "w", encoding="utf-8") as f:
        f.write(final_content)
        f.flush()
        os.fsync(f.fileno())

    print(f"✅ Success! Report saved to: {filename}")
    print(f"📍 Full Path: {full_path}")

    # --- UPDATE LONG-TERM HISTORY ONLY ON SUCCESSFUL EXPORT ---
    # This ensures articles are only "marked as seen" if the file was actually created
    if 'session_seen_urls' in locals() and session_seen_urls:
        update_history_file(session_seen_urls)

except Exception as e:
    print(f"❌ Error during primary save: {e}")

    # Fallback: Save raw data if the Agent formatting failed
    print("\n🔄 Attempting fallback save with raw storage data...")
    try:
        if APPROVED_STORIES_STORAGE.get("approved_stories"):
            fallback_content = f"# Analytics Report (Fallback)\n\nGenerated: {datetime.now(eastern_tz).strftime('%Y-%m-%d %H:%M %Z')}\n\n"
            for article in APPROVED_STORIES_STORAGE["approved_stories"]["evaluated_articles"]:
                fallback_content += f"## [{article['title']}]({article['url']})\n"
                fallback_content += f"- **Score**: {article['score']}/5\n"
                fallback_content += f"- **Date Published**: {article['date_published']}\n"
                fallback_content += f"- **Summary**: {article['summary']}\n\n---\n\n"

            with open(full_path, "w", encoding="utf-8") as f:
                f.write(fallback_content)
                f.flush()
                os.fsync(f.fileno())
            print(f"✅ Fallback save successful!")
    except Exception as fallback_error:
        print(f"❌ Fallback also failed: {fallback_error}")

# Final verification
print("\n🔍 File verification:")
!ls -lh "{full_path}"
print("Executed at:", datetime.now(eastern_tz))

✅ Success! Report saved to: 2025-12-23_1111_Analytics_Report.md
📍 Full Path: /content/drive/MyDrive/AI/Obsidian Vault/daily news/2025-12-23_1111_Analytics_Report.md
💾 History updated: 18 total articles now tracked.

🔍 File verification:
-rw------- 1 root root 4.1K Dec 23 16:11 '/content/drive/MyDrive/AI/Obsidian Vault/daily news/2025-12-23_1111_Analytics_Report.md'
Executed at: 2025-12-23 11:11:14.456613-05:00


In [ ]:
# !rm -rf /content/drive